# Pseudobulk production quality control

This report reads Snakemake models.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
from pathlib import Path
import csv
import pandas as pd
import matplotlib.pyplot as plt
from pyprojroot import here

## Settings

In [ ]:
ROOT = Path(here())
CONFIG = snakemake.config
DATASETS = list(snakemake.params.datasets)
PROD = Path(snakemake.params.mod_root)
OUT = Path(snakemake.params.out_dir)
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def csv_shape(path):
    # Count rows as bytes so QC never parses/materializes large numeric model matrices.
    with open(path, 'rb') as handle:
        header_line = handle.readline()
        n_rows = 0
        while chunk := handle.read(8 * 1024 * 1024):
            n_rows += chunk.count(b'\n')
    with open(path, newline='') as handle:
        header = next(csv.reader(handle))
    return n_rows, len(header) - int(bool(header) and header[0] == '')

rows = []; model_rows = []
methods = [Path(rel_path).parent.name for rel_path in snakemake.params.methods.values()]
for dataset in DATASETS:
    dataset_cfg = CONFIG['datasets'][dataset]
    is_built = bool(dataset_cfg.get('build_pseudobulk', False))
    pb_dir = (PROD / dataset / 'pseudobulk') if is_built else (ROOT / dataset_cfg['pseudobulk_dir'])
    pseudobulk_genes, pseudobulk_samples = csv_shape(pb_dir / 'bulk_expr.csv')
    rank = pd.read_csv(PROD / dataset / 'preprocessing/k.csv')['k'].iloc[0]
    with open(PROD / dataset / 'single_cell_projection/projection_summary.csv', newline='') as handle:
        projection = next(csv.DictReader(handle))
    model_complete = 0
    for method in methods:
        b_path = PROD / dataset / 'models' / method / 'B.csv'
        z_path = PROD / dataset / 'models' / method / 'Z.csv'
        complete = b_path.exists() and z_path.exists(); model_complete += int(complete)
        b_rows, b_cols = csv_shape(b_path) if b_path.exists() else (None, None)
        z_rows, z_cols = csv_shape(z_path) if z_path.exists() else (None, None)
        model_rows.append({'dataset':dataset,'method':method,'complete':complete,
          'B_rows':b_rows,'B_cols':b_cols,'Z_rows':z_rows,'Z_cols':z_cols})
    rows.append({
        'dataset': dataset, 'pseudobulk_source': 'raw-built' if is_built else 'lab-provided',
        'pseudobulk_samples': pseudobulk_samples,
        'pseudobulk_genes': pseudobulk_genes, 'rank': int(rank),
        'models_complete': model_complete, 'models_expected': len(methods),
        'raw_cells': int(projection['n_cells_raw']),
        'projected_cells': int(projection['n_cells_projected']),
        'mapped_cells': int(projection['n_cells_mapped']),
        'projection_passes': int(projection['passes']),
    })
qc = pd.DataFrame(rows)
qc.to_csv(OUT / 'model_building_qc.csv', index=False)
pd.DataFrame(model_rows).to_csv(OUT / 'model_matrix_qc.csv', index=False)
qc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].barh(qc.dataset, qc.pseudobulk_samples)
axes[0].set_xlabel('Pseudobulk samples')
axes[0].set_title('Pseudobulk inputs')
axes[1].barh(qc.dataset, qc.projected_cells / 1e6)
axes[1].set_xlabel('Projected cells (millions)')
axes[1].set_title('All-cell CLAMP projection')
fig.tight_layout()
plt.show()